# Data Analytics and Visualization in Data-Centric Engineering

This workbook contains the worked solution for the workshop. Use it to check the method, the code, and the reasoning behind each step.

The example uses three months of hourly wind turbine telemetry to show cleaning, visualisation, normalisation, and decision-making.

The case study compares three turbines:

- WT-01: healthy and stable
- WT-02: failed in Month 1 and then flatlined
- WT-03: healthy at first, then begins to fail in Month 3

Before starting, run the mock telemetry generation script so that `wind_turbine_telemetry.csv` is in the same folder as this notebook.

## Step 0: Mock Data Setup

Run the generator script first. It creates the dataset used throughout the exercises and includes the failure patterns discussed in the workshop.

## Section 1: Data Ingestion & Data Cleaning [15 mins]

Start with the raw telemetry and decide what should be kept, cleaned, or set aside. In practice, missing values and flatlines are often part of the story rather than simple noise.

A row with missing values might mean a failed sensor, a broken data path, or a machine that has stopped. If we remove those rows too early, we lose useful evidence.

Keep the raw records, then create a cleaned analysis set for the charts and summary statistics.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

csv_candidates = [
    Path('wind_turbine_telemetry.csv'),
    Path('..') / 'wind_turbine_telemetry.csv',
    Path.cwd() / 'wind_turbine_telemetry.csv',
]

for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find wind_turbine_telemetry.csv. Run the mock data script first.')

df = pd.read_csv(csv_path, parse_dates=['Timestamp'])
df.info()
#df.describe(numeric_only=True)

<class 'pandas.DataFrame'>
RangeIndex: 6480 entries, 0 to 6479
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Timestamp              6480 non-null   datetime64[us]
 1   Turbine_ID             6480 non-null   str           
 2   Ambient_Temperature_C  6480 non-null   float64       
 3   Wind_Speed_mps         6480 non-null   float64       
 4   Bearing_Temperature_C  5054 non-null   float64       
 5   Vibration_mm_s         5053 non-null   float64       
 6   RPM                    6480 non-null   float64       
 7   Power_kW               6480 non-null   float64       
 8   Yaw_Error_deg          5064 non-null   float64       
 9   Gearbox_Temperature_C  5064 non-null   float64       
 10  Turbine_Load_Percent   6480 non-null   float64       
 11  Operational_State      6480 non-null   str           
 12  Maintenance_Flag       6480 non-null   str           
 13  Grid_Priority 

In [2]:
df['Failure_State'] = np.where(
    df[['Bearing_Temperature_C', 'Vibration_mm_s']].isna().any(axis=1),
    'Sensor dropout / failure',
    'Normal',
)

failure_records = df[df['Failure_State'] == 'Sensor dropout / failure'].copy()
analysis_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s']).copy()

print('Failure records retained for forensic review:', len(failure_records))
print('Rows available for statistical analysis:', len(analysis_df))
analysis_df.head()

Failure records retained for forensic review: 1437
Rows available for statistical analysis: 5043


,Timestamp,Turbine_ID,Ambient_Temperature_C,Wind_Speed_mps,Bearing_Temperature_C,Vibration_mm_s,RPM,Power_kW,Yaw_Error_deg,Gearbox_Temperature_C,Turbine_Load_Percent,Operational_State,Maintenance_Flag,Grid_Priority,Failure_State
0,2026-01-01 00:00:00,WT-01,12.58,6.59,50.50,2.21,13.65,1427.5,1.00,58.63,73.3,normal,none,Standard,Normal
1,2026-01-01 01:00:00,WT-01,10.97,5.95,49.01,2.09,13.54,1329.0,0.56,57.54,70.7,normal,none,Standard,Normal
2,2026-01-01 02:00:00,WT-01,11.84,7.63,51.24,2.19,13.63,1586.6,1.28,59.62,80.5,normal,none,Standard,Normal
3,2026-01-01 03:00:00,WT-01,11.42,8.36,50.76,2.22,13.74,1707.4,1.15,59.44,85.0,normal,none,Standard,Normal
4,2026-01-01 04:00:00,WT-01,8.58,8.96,48.17,2.18,13.66,1809.3,1.09,57.46,88.7,normal,none,Standard,Normal


## Section 2: Interactive Anomaly Hunting via Scatter Plots [15 mins]

Temperature, vibration, and RPM often move together in ways that say something about asset health. Interactive plots make it easier to inspect the relationships and spot when a turbine starts to drift away from normal behaviour.

Use hover and zoom to inspect the point where behaviour changes.

In [4]:
import plotly.express as px

scatter_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM']).copy()

fig = px.scatter(
    scatter_df,
    x='Bearing_Temperature_C',
    y='Vibration_mm_s',
    color='Turbine_ID',
#    size='RPM',
    size_max=6,
    hover_data=['Timestamp', 'RPM', 'Grid_Priority'],
    title='Vibration vs Bearing Temperature by Turbine',
)
fig.update_layout(template='plotly_white')
fig

In [5]:
temperature_df = df.sort_values(['Turbine_ID', 'Timestamp']).copy()

fig = px.line(
    temperature_df,
    x='Timestamp',
    y='Bearing_Temperature_C',
    color='Turbine_ID',
    markers=True,
    title='Bearing Temperature Over the 3-Month Timeline',
    hover_data={'Vibration_mm_s': True, 'RPM': True, 'Grid_Priority': True},
)
fig.update_layout(template='plotly_white', xaxis_title='Timestamp', yaxis_title='Bearing Temperature (C)')
fig

### Interactive Challenge

Use Plotly's zoom and hover tools to find when WT-02 stops producing useful telemetry and when WT-03 starts to move away from the healthy pattern. The point is to connect the chart back to the asset, not just describe its shape.

## Section 3: Vibration Normalised by RPM [15 mins]

Rather than rescaling every feature separately, compare vibration against RPM. This gives a simple view of how much vibration the turbine is producing for a given operating speed.

A rising vibration-to-RPM ratio can be a useful warning sign when a machine starts to behave less smoothly.

In [8]:
wt03_df = df[df['Turbine_ID'] == 'WT-03'].sort_values('Timestamp').copy()

wt03_df['Vibration_per_RPM'] = wt03_df['Vibration_mm_s'] / wt03_df['RPM'].replace(0, np.nan)

fig = px.line(
    wt03_df,
    x='Timestamp',
    y='Vibration_per_RPM',
    markers=True,
    title='WT-03 Vibration Normalised by RPM',
    hover_data=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM'],
)
fig.update_layout(
    template='plotly_white',
    xaxis_title='Timestamp',
    yaxis_title='Vibration per RPM',
)
fig

## Section 4: The Operational Decision & Dashboard Ethics [15 mins]

Dashboards are not neutral. They influence maintenance timing, budget decisions, and safety risk. Use the chart to show which turbine should be prioritised if only one overhaul can be funded this quarter.

WT-01 is healthy, WT-02 has already failed, and WT-03 is the one still moving in the wrong direction. The chart should make the trade-off obvious.

In [7]:
latest_rows = df.sort_values('Timestamp').groupby('Turbine_ID').tail(1).copy()

remaining_lifespan = {
    'WT-01': 120,
    'WT-02': 0,
    'WT-03': 14,
}

latest_rows['Remaining_Operational_Days'] = latest_rows['Turbine_ID'].map(remaining_lifespan)
latest_rows['Priority_Label'] = latest_rows['Grid_Priority']

fig = px.bar(
    latest_rows,
    x='Turbine_ID',
    y='Remaining_Operational_Days',
    color='Priority_Label',
    text='Remaining_Operational_Days',
    hover_data=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM'],
    title='Executive Dashboard: Remaining Operational Lifespan by Turbine',
)
fig.update_layout(template='plotly_white', yaxis_title='Estimated Remaining Operational Days')
fig

### Ethics & Governance Challenge

Consider the full cost of the decision. Do you spend the budget replacing WT-02, which is already offline and represents lost revenue, or do you intervene on WT-03 before it fails in the same way?

Your answer should cover safety, environmental impact, operations, and liability. Data quality, visualisation design, and maintenance policy all feed into that choice.

## Wrap-Up

By the end of this workbook, you should be able to move from raw telemetry to a defensible operational recommendation using cleaning, plotting, normalisation, and a simple decision dashboard.